# Práctica 5. Búsqueda heurística: voraz y A*

**Asignatura:** Inteligencia Artificial
**Unidad:** 2. Resolución de problemas por métodos de búsqueda

## Objetivo

Implementar búsqueda voraz y A* con diferentes heurísticas y analizar su optimalidad y eficiencia.

La práctica se divide en dos partes:

1. Búsqueda heurística sobre un mapa de carreteras.
2. Resolución del problema 8-puzzle mediante diferentes heurísticas.

## Grafo del mapa
Reutilizamos el mismo mapa que habíamos creado en la Práctica 4

In [1]:
# ---------------------------------------------------------
# MAPA DE CARRETERAS
# ---------------------------------------------------------
# Reutilizamos el mismo mapa empleado en la Práctica 4.
#
# Cada ciudad representa un estado.
# Cada conexión representa una carretera.
# El valor asociado representa la distancia en kilómetros.
#
# Las carreteras pueden recorrerse en ambos sentidos.
# ---------------------------------------------------------

grafo = {

    "Morelia": {
        "Querétaro": 195,
        "Toluca": 245
    },

    "Querétaro": {
        "CDMX": 213,
        "Morelia": 195,
        "Pachuca": 240,
        "Toluca": 190
    },

    "Toluca": {
        "CDMX": 66,
        "Cuernavaca": 110,
        "Morelia": 245,
        "Querétaro": 190
    },

    "CDMX": {
        "Cuernavaca": 89,
        "Pachuca": 95,
        "Puebla": 130,
        "Querétaro": 213,
        "Toluca": 66
    },

    "Pachuca": {
        "CDMX": 95,
        "Querétaro": 240,
        "Tlaxcala": 150
    },

    "Cuernavaca": {
        "CDMX": 89,
        "Puebla": 160,
        "Toluca": 110
    },

    "Tlaxcala": {
        "Pachuca": 150,
        "Puebla": 35
    },

    "Puebla": {
        "CDMX": 130,
        "Cuernavaca": 160,
        "Tlaxcala": 35
    }
}

## Estado inicial y meta
### Morelia → Puebla

In [2]:
# ---------------------------------------------------------
# ESTADO INICIAL Y ESTADO META
# ---------------------------------------------------------

# Ciudad donde comienza la búsqueda.
estado_inicial = "Morelia"

# Ciudad que queremos alcanzar.
estado_meta = "Puebla"

# Comprobamos los valores definidos.
print("Estado inicial:", estado_inicial)
print("Estado meta:", estado_meta)

Estado inicial: Morelia
Estado meta: Puebla


## Heurística

In [3]:
# ---------------------------------------------------------
# FUNCIÓN HEURÍSTICA
# ---------------------------------------------------------
# La heurística h(n) representa una estimación de la
# distancia en línea recta desde cada ciudad hasta Puebla.
#
# Un valor menor indica que, según la estimación,
# la ciudad se encuentra más cerca de la meta.
# ---------------------------------------------------------

heuristica = {
    "Morelia": 330,
    "Querétaro": 210,
    "Toluca": 160,
    "CDMX": 110,
    "Pachuca": 120,
    "Cuernavaca": 130,
    "Tlaxcala": 30,
    "Puebla": 0
}

## Visualizar los valores

In [4]:
# ---------------------------------------------------------
# COMPROBACIÓN DE LA HEURÍSTICA
# ---------------------------------------------------------

# Recorremos las ciudades y mostramos el valor heurístico
# asociado a cada una.

for ciudad, valor_h in heuristica.items():
    print(f"h({ciudad}) = {valor_h} km")

h(Morelia) = 330 km
h(Querétaro) = 210 km
h(Toluca) = 160 km
h(CDMX) = 110 km
h(Pachuca) = 120 km
h(Cuernavaca) = 130 km
h(Tlaxcala) = 30 km
h(Puebla) = 0 km


## Importar heapq

In [5]:
# Importamos heapq para utilizar una cola de prioridad.
# La necesitaremos para calcular los costos óptimos reales
# mediante búsqueda de costo uniforme (UCS).

import heapq

## Función auxiliar para calcular el costo óptimo
Esta versión de UCS es más sencilla que la de P4 porque ahora solo necesitamos conocer el costo óptimo, no la ruta completa.

In [6]:
# ---------------------------------------------------------
# FUNCIÓN PARA CALCULAR EL COSTO ÓPTIMO HASTA LA META
# ---------------------------------------------------------
# Utilizamos búsqueda de costo uniforme (UCS) para obtener
# el costo real mínimo desde una ciudad hasta Puebla.
#
# Este costo servirá para comprobar si la heurística
# proporcionada es admisible.
# ---------------------------------------------------------

def costo_optimo(grafo, inicio, meta):

    # La frontera funciona como una cola de prioridad.
    #
    # Cada elemento contiene:
    # (costo_acumulado, ciudad_actual)
    frontera = [
        (0, inicio)
    ]

    # Guardamos el menor costo conocido para llegar
    # a cada ciudad.
    mejores_costos = {
        inicio: 0
    }

    # Mientras existan ciudades pendientes...
    while frontera:

        # Extraemos la ciudad con menor costo acumulado.
        costo, ciudad_actual = heapq.heappop(frontera)

        # Si llegamos a la meta, este costo es óptimo.
        if ciudad_actual == meta:
            return costo

        # Si existe una ruta mejor hacia esta misma ciudad,
        # ignoramos esta entrada.
        if costo > mejores_costos[ciudad_actual]:
            continue

        # Exploramos todas las ciudades vecinas.
        for vecino, distancia in grafo[ciudad_actual].items():

            # Calculamos el costo de llegar al vecino
            # mediante la ruta actual.
            nuevo_costo = costo + distancia

            # Actualizamos únicamente si encontramos
            # una ruta más barata.
            if (
                vecino not in mejores_costos
                or nuevo_costo < mejores_costos[vecino]
            ):

                mejores_costos[vecino] = nuevo_costo

                # Agregamos el nuevo costo a la
                # cola de prioridad.
                heapq.heappush(
                    frontera,
                    (nuevo_costo, vecino)
                )

    # Si no existe una ruta hacia la meta.
    return None

## Comprobar admisibilidad

In [7]:
# ---------------------------------------------------------
# COMPROBACIÓN DE ADMISIBILIDAD
# ---------------------------------------------------------

# Suponemos inicialmente que la heurística es admisible.
es_admisible = True

# Recorremos todas las ciudades.
for ciudad in heuristica:

    # Calculamos el costo óptimo real desde esta ciudad
    # hasta Puebla.
    costo_real = costo_optimo(
        grafo,
        ciudad,
        estado_meta
    )

    # Obtenemos la estimación heurística de la ciudad.
    valor_h = heuristica[ciudad]

    # Comprobamos la condición:
    #
    # h(n) <= h*(n)
    cumple = valor_h <= costo_real

    # Mostramos la comparación.
    print(
        f"{ciudad}: "
        f"h(n) = {valor_h}, "
        f"h*(n) = {costo_real}, "
        f"¿Cumple? {cumple}"
    )

    # Si alguna ciudad sobreestima el costo real,
    # la heurística completa deja de ser admisible.
    if not cumple:
        es_admisible = False

# Mostramos la conclusión final.
print("\n¿La heurística es admisible?", es_admisible)

Morelia: h(n) = 330, h*(n) = 441, ¿Cumple? True
Querétaro: h(n) = 210, h*(n) = 343, ¿Cumple? True
Toluca: h(n) = 160, h*(n) = 196, ¿Cumple? True
CDMX: h(n) = 110, h*(n) = 130, ¿Cumple? True
Pachuca: h(n) = 120, h*(n) = 185, ¿Cumple? True
Cuernavaca: h(n) = 130, h*(n) = 160, ¿Cumple? True
Tlaxcala: h(n) = 30, h*(n) = 35, ¿Cumple? True
Puebla: h(n) = 0, h*(n) = 0, ¿Cumple? True

¿La heurística es admisible? True


## Comprobar consistencia

In [8]:
# ---------------------------------------------------------
# COMPROBACIÓN DE CONSISTENCIA
# ---------------------------------------------------------

# Suponemos inicialmente que la heurística es consistente.
es_consistente = True

# Recorremos cada ciudad del grafo.
for ciudad in grafo:

    # Revisamos cada conexión de la ciudad.
    for vecino, distancia in grafo[ciudad].items():

        # Comprobamos la condición de consistencia:
        #
        # h(n) <= c(n, n') + h(n')
        cumple = (
            heuristica[ciudad]
            <= distancia + heuristica[vecino]
        )

        # Mostramos la comprobación realizada.
        print(
            f"{ciudad} -> {vecino}: "
            f"{heuristica[ciudad]} <= "
            f"{distancia} + {heuristica[vecino]} "
            f"¿Cumple? {cumple}"
        )

        # Basta con que falle una conexión para que
        # la heurística no sea consistente.
        if not cumple:
            es_consistente = False

# Mostramos la conclusión final.
print("\n¿La heurística es consistente?", es_consistente)

Morelia -> Querétaro: 330 <= 195 + 210 ¿Cumple? True
Morelia -> Toluca: 330 <= 245 + 160 ¿Cumple? True
Querétaro -> CDMX: 210 <= 213 + 110 ¿Cumple? True
Querétaro -> Morelia: 210 <= 195 + 330 ¿Cumple? True
Querétaro -> Pachuca: 210 <= 240 + 120 ¿Cumple? True
Querétaro -> Toluca: 210 <= 190 + 160 ¿Cumple? True
Toluca -> CDMX: 160 <= 66 + 110 ¿Cumple? True
Toluca -> Cuernavaca: 160 <= 110 + 130 ¿Cumple? True
Toluca -> Morelia: 160 <= 245 + 330 ¿Cumple? True
Toluca -> Querétaro: 160 <= 190 + 210 ¿Cumple? True
CDMX -> Cuernavaca: 110 <= 89 + 130 ¿Cumple? True
CDMX -> Pachuca: 110 <= 95 + 120 ¿Cumple? True
CDMX -> Puebla: 110 <= 130 + 0 ¿Cumple? True
CDMX -> Querétaro: 110 <= 213 + 210 ¿Cumple? True
CDMX -> Toluca: 110 <= 66 + 160 ¿Cumple? True
Pachuca -> CDMX: 120 <= 95 + 110 ¿Cumple? True
Pachuca -> Querétaro: 120 <= 240 + 210 ¿Cumple? True
Pachuca -> Tlaxcala: 120 <= 150 + 30 ¿Cumple? True
Cuernavaca -> CDMX: 130 <= 89 + 110 ¿Cumple? True
Cuernavaca -> Puebla: 130 <= 160 + 0 ¿Cumple? Tru

## Búsqueda voraz
En esta parte heuristica[vecino] heapq ordenará las opciones según h(n).

La variable nuevo_costo no constituye el criterio principal de búsqueda.

In [9]:
# ---------------------------------------------------------
# BÚSQUEDA VORAZ (GREEDY BEST-FIRST SEARCH)
# ---------------------------------------------------------
# La búsqueda voraz selecciona para expansión el nodo
# que tenga el MENOR valor heurístico h(n).
#
# La heurística estima qué tan cerca se encuentra una
# ciudad de Puebla.
#
# IMPORTANTE:
# La búsqueda voraz NO utiliza el costo acumulado para
# decidir qué nodo explorar primero.
# ---------------------------------------------------------

def busqueda_voraz(grafo, heuristica, inicio, meta):

    # La frontera será una cola de prioridad.
    #
    # Cada elemento contiene:
    # (valor_heuristico, ciudad_actual, ruta, costo_acumulado)
    #
    # El valor heurístico aparece primero porque heapq
    # utilizará ese valor como prioridad.
    frontera = [
        (heuristica[inicio], inicio, [inicio], 0)
    ]

    # Guardamos las ciudades que ya fueron expandidas
    # para evitar procesarlas repetidamente.
    visitados = set()

    # Contador de nodos expandidos.
    nodos_expandidos = 0

    # Mientras existan ciudades pendientes por explorar...
    while frontera:

        # Extraemos el elemento con el MENOR valor h(n).
        valor_h, ciudad_actual, ruta, costo = heapq.heappop(
            frontera
        )

        # Si la ciudad ya fue procesada, la ignoramos.
        if ciudad_actual in visitados:
            continue

        # Si llegamos a Puebla, devolvemos los resultados.
        if ciudad_actual == meta:
            return ruta, costo, nodos_expandidos

        # Marcamos la ciudad como visitada.
        visitados.add(ciudad_actual)

        # Como exploraremos sus vecinos,
        # contamos este nodo como expandido.
        nodos_expandidos += 1

        # Recorremos los vecinos en orden alfabético.
        for vecino in sorted(grafo[ciudad_actual]):

            # Solo consideramos ciudades que todavía
            # no hayan sido expandidas.
            if vecino not in visitados:

                # Obtenemos la distancia de la carretera.
                distancia = grafo[ciudad_actual][vecino]

                # Calculamos el costo acumulado de la ruta.
                #
                # OJO:
                # Este costo se guarda para reportarlo,
                # pero NO determina la prioridad de búsqueda.
                nuevo_costo = costo + distancia

                # Construimos la nueva ruta.
                nueva_ruta = ruta + [vecino]

                # Agregamos el vecino a la cola.
                #
                # Su prioridad depende solamente de h(n).
                heapq.heappush(
                    frontera,
                    (
                        heuristica[vecino],
                        vecino,
                        nueva_ruta,
                        nuevo_costo
                    )
                )

    # Si agotamos la frontera sin encontrar Puebla,
    # no existe una solución.
    return None, None, nodos_expandidos

## Ejecutar búsqueda voraz

In [10]:
# ---------------------------------------------------------
# EJECUCIÓN DE BÚSQUEDA VORAZ
# ---------------------------------------------------------

# Ejecutamos la búsqueda voraz desde Morelia hasta Puebla.
ruta_voraz, costo_voraz, nodos_voraz = busqueda_voraz(
    grafo,
    heuristica,
    estado_inicial,
    estado_meta
)

# Mostramos los resultados.
print("RESULTADOS DE BÚSQUEDA VORAZ")
print("----------------------------")

print("Ruta encontrada:", " -> ".join(ruta_voraz))
print("Número de tramos:", len(ruta_voraz) - 1)
print("Costo total:", costo_voraz, "km")
print("Nodos expandidos:", nodos_voraz)

RESULTADOS DE BÚSQUEDA VORAZ
----------------------------
Ruta encontrada: Morelia -> Toluca -> CDMX -> Puebla
Número de tramos: 3
Costo total: 441 km
Nodos expandidos: 3


## Algoritmo A*
A* se pregunta ¿Cuánto me ha costado llegar hasta aquí + cuánto estimo que todavía me falta?

In [11]:
# ---------------------------------------------------------
# BÚSQUEDA A*
# ---------------------------------------------------------
# A* selecciona para expansión el nodo con el menor:
#
#               f(n) = g(n) + h(n)
#
# donde:
#
# g(n) = costo real acumulado desde el estado inicial.
# h(n) = costo estimado desde el nodo hasta la meta.
# f(n) = costo total estimado de la solución.
# ---------------------------------------------------------

def a_estrella(grafo, heuristica, inicio, meta):

    # La frontera será una cola de prioridad.
    #
    # Cada elemento contiene:
    # (f(n), g(n), ciudad_actual, ruta)
    #
    # Para el estado inicial:
    # g(inicio) = 0
    # f(inicio) = 0 + h(inicio)
    frontera = [
        (
            heuristica[inicio],
            0,
            inicio,
            [inicio]
        )
    ]

    # Guardamos el menor costo g(n) conocido
    # para llegar a cada ciudad.
    mejores_costos = {
        inicio: 0
    }

    # Contador de nodos expandidos.
    nodos_expandidos = 0

    # Mientras existan nodos pendientes...
    while frontera:

        # Extraemos el nodo que tenga el menor valor f(n).
        valor_f, costo_g, ciudad_actual, ruta = heapq.heappop(
            frontera
        )

        # Si llegamos a la meta, devolvemos la solución.
        if ciudad_actual == meta:
            return ruta, costo_g, nodos_expandidos

        # Si ya conocemos una ruta más barata para llegar
        # a esta misma ciudad, ignoramos esta entrada.
        if costo_g > mejores_costos[ciudad_actual]:
            continue

        # Vamos a explorar sus vecinos.
        nodos_expandidos += 1

        # Recorremos los vecinos en orden alfabético.
        for vecino in sorted(grafo[ciudad_actual]):

            # Obtenemos el costo de la carretera.
            distancia = grafo[ciudad_actual][vecino]

            # Calculamos g(n):
            # costo real acumulado hasta el vecino.
            nuevo_costo_g = costo_g + distancia

            # Comprobamos si nunca habíamos llegado al vecino
            # o acabamos de encontrar una ruta más barata.
            if (
                vecino not in mejores_costos
                or nuevo_costo_g < mejores_costos[vecino]
            ):

                # Actualizamos el mejor costo conocido.
                mejores_costos[vecino] = nuevo_costo_g

                # Calculamos:
                #
                # f(n) = g(n) + h(n)
                nuevo_valor_f = (
                    nuevo_costo_g
                    + heuristica[vecino]
                )

                # Construimos la nueva ruta.
                nueva_ruta = ruta + [vecino]

                # Agregamos el nuevo estado a la
                # cola de prioridad.
                heapq.heappush(
                    frontera,
                    (
                        nuevo_valor_f,
                        nuevo_costo_g,
                        vecino,
                        nueva_ruta
                    )
                )

    # Si no existe una solución.
    return None, None, nodos_expandidos

## Ejecutar A*

In [12]:
# ---------------------------------------------------------
# EJECUCIÓN DE A*
# ---------------------------------------------------------

# Ejecutamos A* desde Morelia hasta Puebla.
ruta_a, costo_a, nodos_a = a_estrella(
    grafo,
    heuristica,
    estado_inicial,
    estado_meta
)

# Mostramos los resultados.
print("RESULTADOS DE A*")
print("----------------")

print("Ruta encontrada:", " -> ".join(ruta_a))
print("Número de tramos:", len(ruta_a) - 1)
print("Costo total:", costo_a, "km")
print("Nodos expandidos:", nodos_a)

RESULTADOS DE A*
----------------
Ruta encontrada: Morelia -> Toluca -> CDMX -> Puebla
Número de tramos: 3
Costo total: 441 km
Nodos expandidos: 4


## UCS para la comparación

In [13]:
# ---------------------------------------------------------
# BÚSQUEDA DE COSTO UNIFORME (UCS)
# ---------------------------------------------------------
# UCS utiliza únicamente el costo real acumulado g(n)
# para decidir qué nodo debe expandirse primero.
#
# A diferencia de:
#
# Voraz: f(n) = h(n)
# A*:    f(n) = g(n) + h(n)
# UCS:   f(n) = g(n)
# ---------------------------------------------------------

def ucs(grafo, inicio, meta):

    # La frontera será una cola de prioridad.
    #
    # Cada elemento contiene:
    # (costo_acumulado, ciudad_actual, ruta)
    frontera = [
        (0, inicio, [inicio])
    ]

    # Guardamos el menor costo conocido para
    # llegar a cada ciudad.
    mejores_costos = {
        inicio: 0
    }

    # Contador de nodos expandidos.
    nodos_expandidos = 0

    # Mientras existan estados pendientes...
    while frontera:

        # Extraemos el estado con menor costo acumulado.
        costo, ciudad_actual, ruta = heapq.heappop(
            frontera
        )

        # Si llegamos a la meta, devolvemos la solución.
        if ciudad_actual == meta:
            return ruta, costo, nodos_expandidos

        # Ignoramos una entrada si ya conocemos
        # una ruta más barata hacia la misma ciudad.
        if costo > mejores_costos[ciudad_actual]:
            continue

        # Expandimos el nodo actual.
        nodos_expandidos += 1

        # Recorremos sus vecinos en orden alfabético.
        for vecino in sorted(grafo[ciudad_actual]):

            # Obtenemos la distancia de la carretera.
            distancia = grafo[ciudad_actual][vecino]

            # Calculamos el nuevo costo acumulado.
            nuevo_costo = costo + distancia

            # Solo actualizamos si encontramos
            # una ruta más barata hacia el vecino.
            if (
                vecino not in mejores_costos
                or nuevo_costo < mejores_costos[vecino]
            ):

                # Guardamos el nuevo mejor costo.
                mejores_costos[vecino] = nuevo_costo

                # Construimos la nueva ruta.
                nueva_ruta = ruta + [vecino]

                # Introducimos el estado en la cola
                # utilizando g(n) como prioridad.
                heapq.heappush(
                    frontera,
                    (
                        nuevo_costo,
                        vecino,
                        nueva_ruta
                    )
                )

    # Si no encontramos una solución.
    return None, None, nodos_expandidos

## Ejecutar UCS

In [14]:
# ---------------------------------------------------------
# EJECUCIÓN DE UCS
# ---------------------------------------------------------

ruta_ucs, costo_ucs, nodos_ucs = ucs(
    grafo,
    estado_inicial,
    estado_meta
)

# Mostramos los resultados.
print("RESULTADOS DE UCS")
print("-----------------")

print("Ruta encontrada:", " -> ".join(ruta_ucs))
print("Número de tramos:", len(ruta_ucs) - 1)
print("Costo total:", costo_ucs, "km")
print("Nodos expandidos:", nodos_ucs)

RESULTADOS DE UCS
-----------------
Ruta encontrada: Morelia -> Toluca -> CDMX -> Puebla
Número de tramos: 3
Costo total: 441 km
Nodos expandidos: 6


## Tabla comparativa

In [15]:
# ---------------------------------------------------------
# TABLA COMPARATIVA - PARTE A
# ---------------------------------------------------------

# Importamos pandas para organizar los resultados
# de los tres algoritmos en una tabla.
import pandas as pd

# Creamos una lista con los resultados obtenidos
# durante las ejecuciones.
resultados_mapa = [
    {
        "Algoritmo": "Voraz",
        "Ruta": " -> ".join(ruta_voraz),
        "Tramos": len(ruta_voraz) - 1,
        "Costo (km)": costo_voraz,
        "Nodos expandidos": nodos_voraz
    },

    {
        "Algoritmo": "A*",
        "Ruta": " -> ".join(ruta_a),
        "Tramos": len(ruta_a) - 1,
        "Costo (km)": costo_a,
        "Nodos expandidos": nodos_a
    },

    {
        "Algoritmo": "UCS",
        "Ruta": " -> ".join(ruta_ucs),
        "Tramos": len(ruta_ucs) - 1,
        "Costo (km)": costo_ucs,
        "Nodos expandidos": nodos_ucs
    }
]

# Convertimos los resultados en un DataFrame.
tabla_mapa = pd.DataFrame(resultados_mapa)

# Mostramos la tabla.
tabla_mapa

,Algoritmo,Ruta,Tramos,Costo (km),Nodos expandidos
0,Voraz,Morelia -> Toluca -> CDMX -> Puebla,3,441,3
1,A*,Morelia -> Toluca -> CDMX -> Puebla,3,441,4
2,UCS,Morelia -> Toluca -> CDMX -> Puebla,3,441,6


## Análisis de resultados - Parte A

Los tres algoritmos encontraron la misma ruta:

**Morelia → Toluca → CDMX → Puebla**

con un costo total de **441 km** y 3 tramos.

### ¿Cuál algoritmo expandió menos nodos?

La búsqueda voraz expandió **3 nodos**, A* expandió **4 nodos** y UCS expandió **6 nodos**. Por lo tanto, en este experimento la búsqueda voraz fue la que expandió menos nodos.

Sin embargo, esto no significa que la búsqueda voraz sea siempre mejor. Este algoritmo utiliza únicamente la heurística `h(n)` para decidir qué nodo expandir y no considera el costo acumulado de la ruta. En este problema particular, la heurística logró conducirla directamente hacia la ruta óptima.

### ¿Cuál algoritmo garantiza optimalidad?

La búsqueda voraz **no garantiza optimalidad**, aunque en este experimento haya encontrado la ruta óptima.

UCS sí garantiza encontrar la solución de menor costo cuando los costos de las acciones son positivos.

A* también garantiza la solución óptima bajo las condiciones correspondientes de la heurística. En este problema se comprobó previamente que la heurística utilizada es **admisible y consistente**.

### Diferencia entre Voraz, A* y UCS

Los tres algoritmos utilizan criterios distintos para seleccionar el siguiente nodo:

- **Voraz:** utiliza únicamente `h(n)`, es decir, la estimación del costo restante hasta la meta.
- **UCS:** utiliza únicamente `g(n)`, es decir, el costo real acumulado desde el estado inicial.
- **A*:** combina ambos valores mediante `f(n) = g(n) + h(n)`.

Los resultados muestran que una buena heurística puede reducir la cantidad de nodos que es necesario expandir. Sin embargo, encontrar una solución óptima en una ejecución particular no significa que el algoritmo tenga garantía general de optimalidad.

# Parte B. 8-puzzle

En esta parte se resolverá el problema del 8-puzzle utilizando el algoritmo A*.

El estado inicial es:

2  8  3
1  6  4
7  _  5

El estado meta es:

1  2  3
8  _  4
7  6  5

Cada movimiento tiene un costo de 1.

## Estados inicial y meta

In [16]:
# ---------------------------------------------------------
# ESTADOS DEL 8-PUZZLE
# ---------------------------------------------------------
# Representamos cada tablero mediante una tupla de
# 9 elementos.
#
# Los números del 1 al 8 representan las fichas.
# El número 0 representa el espacio vacío.
# ---------------------------------------------------------

estado_inicial_puzzle = (
    2, 8, 3,
    1, 6, 4,
    7, 0, 5
)

estado_meta_puzzle = (
    1, 2, 3,
    8, 0, 4,
    7, 6, 5
)

## Función para mostrar un tablero

In [17]:
# ---------------------------------------------------------
# FUNCIÓN PARA MOSTRAR UN TABLERO
# ---------------------------------------------------------

def mostrar_tablero(estado):

    # Recorremos las posiciones del tablero
    # de tres en tres para formar cada fila.
    for i in range(0, 9, 3):

        # Tomamos tres posiciones consecutivas.
        fila = estado[i:i + 3]

        # Sustituimos visualmente el 0 por "_".
        fila_visible = [
            "_" if valor == 0 else str(valor)
            for valor in fila
        ]

        # Mostramos la fila separando sus elementos.
        print(" ".join(fila_visible))

## Comprobar ambos estados

In [18]:
# ---------------------------------------------------------
# COMPROBACIÓN DE LOS ESTADOS
# ---------------------------------------------------------

print("ESTADO INICIAL")
mostrar_tablero(estado_inicial_puzzle)

print("\nESTADO META")
mostrar_tablero(estado_meta_puzzle)

ESTADO INICIAL
2 8 3
1 6 4
7 _ 5

ESTADO META
1 2 3
8 _ 4
7 6 5


## Generar sucesores

In [19]:
# ---------------------------------------------------------
# GENERACIÓN DE SUCESORES DEL 8-PUZZLE
# ---------------------------------------------------------
# Esta función recibe un estado del puzzle y genera
# todos los estados que pueden alcanzarse mediante
# un movimiento válido del espacio vacío.
#
# Cada sucesor contendrá:
#   1. El nombre del movimiento realizado.
#   2. El nuevo estado resultante.
# ---------------------------------------------------------

def obtener_sucesores(estado):

    # Buscamos la posición donde se encuentra el espacio
    # vacío, representado internamente mediante el número 0.
    posicion_vacia = estado.index(0)

    # Convertimos la posición de la tupla en
    # coordenadas de fila y columna.
    #
    # Ejemplo:
    # posición 7 -> fila 2, columna 1
    fila = posicion_vacia // 3
    columna = posicion_vacia % 3

    # Aquí almacenaremos todos los movimientos válidos
    # encontrados desde el estado actual.
    sucesores = []

    # -----------------------------------------------------
    # MOVIMIENTO: ARRIBA
    # -----------------------------------------------------
    # El vacío puede subir solamente si no se encuentra
    # actualmente en la primera fila.
    if fila > 0:

        # La posición que está arriba se encuentra
        # tres lugares antes dentro de la tupla.
        nueva_posicion = posicion_vacia - 3

        # Convertimos temporalmente la tupla en lista
        # porque las tuplas no pueden modificarse.
        nuevo_estado = list(estado)

        # Intercambiamos el espacio vacío con la ficha
        # situada arriba.
        nuevo_estado[posicion_vacia], nuevo_estado[nueva_posicion] = (
            nuevo_estado[nueva_posicion],
            nuevo_estado[posicion_vacia]
        )

        # Volvemos a convertir el estado en tupla
        # y lo guardamos junto con el movimiento realizado.
        sucesores.append(
            ("Arriba", tuple(nuevo_estado))
        )

    # -----------------------------------------------------
    # MOVIMIENTO: ABAJO
    # -----------------------------------------------------
    # El vacío puede bajar solamente si no está
    # en la última fila.
    if fila < 2:

        nueva_posicion = posicion_vacia + 3
        nuevo_estado = list(estado)

        nuevo_estado[posicion_vacia], nuevo_estado[nueva_posicion] = (
            nuevo_estado[nueva_posicion],
            nuevo_estado[posicion_vacia]
        )

        sucesores.append(
            ("Abajo", tuple(nuevo_estado))
        )

    # -----------------------------------------------------
    # MOVIMIENTO: IZQUIERDA
    # -----------------------------------------------------
    # El vacío puede moverse a la izquierda solamente
    # si no se encuentra en la primera columna.
    if columna > 0:

        nueva_posicion = posicion_vacia - 1
        nuevo_estado = list(estado)

        nuevo_estado[posicion_vacia], nuevo_estado[nueva_posicion] = (
            nuevo_estado[nueva_posicion],
            nuevo_estado[posicion_vacia]
        )

        sucesores.append(
            ("Izquierda", tuple(nuevo_estado))
        )

    # -----------------------------------------------------
    # MOVIMIENTO: DERECHA
    # -----------------------------------------------------
    # El vacío puede moverse a la derecha solamente
    # si no se encuentra en la última columna.
    if columna < 2:

        nueva_posicion = posicion_vacia + 1
        nuevo_estado = list(estado)

        nuevo_estado[posicion_vacia], nuevo_estado[nueva_posicion] = (
            nuevo_estado[nueva_posicion],
            nuevo_estado[posicion_vacia]
        )

        sucesores.append(
            ("Derecha", tuple(nuevo_estado))
        )

    # Devolvemos todos los movimientos posibles.
    return sucesores

## Probar los sucesores

In [20]:
# ---------------------------------------------------------
# COMPROBACIÓN DE LOS MOVIMIENTOS POSIBLES
# ---------------------------------------------------------

# Generamos todos los sucesores posibles
# desde el estado inicial.
sucesores_iniciales = obtener_sucesores(
    estado_inicial_puzzle
)

# Mostramos cuántos movimientos son posibles.
print(
    "Número de movimientos posibles:",
    len(sucesores_iniciales)
)

# Recorremos cada movimiento generado.
for movimiento, estado in sucesores_iniciales:

    # Mostramos qué movimiento realizó el espacio vacío.
    print("\nMovimiento:", movimiento)

    # Mostramos el tablero resultante.
    mostrar_tablero(estado)

Número de movimientos posibles: 3

Movimiento: Arriba
2 8 3
1 _ 4
7 6 5

Movimiento: Izquierda
2 8 3
1 6 4
_ 7 5

Movimiento: Derecha
2 8 3
1 6 4
7 5 _


## Fichas fuera de lugar

In [21]:
# ---------------------------------------------------------
# HEURÍSTICA 1: FICHAS FUERA DE LUGAR
# ---------------------------------------------------------
# Esta heurística cuenta cuántas fichas no se encuentran
# en la posición que les corresponde en el estado meta.
#
# El espacio vacío (0) NO se cuenta como una ficha.
# ---------------------------------------------------------

def fichas_fuera_de_lugar(estado, meta):

    # Inicializamos el contador.
    fuera_de_lugar = 0

    # Recorremos las 9 posiciones del tablero.
    for i in range(9):

        # Comprobamos dos condiciones:
        #
        # 1. La posición no contiene el espacio vacío.
        # 2. La ficha no coincide con la ficha que debería
        #    estar en esa posición según el estado meta.
        if estado[i] != 0 and estado[i] != meta[i]:

            # Encontramos una ficha mal colocada.
            fuera_de_lugar += 1

    # Devolvemos el número total de fichas
    # que están fuera de su posición.
    return fuera_de_lugar

## Distancia Manhattan

In [22]:
# ---------------------------------------------------------
# HEURÍSTICA 2: DISTANCIA MANHATTAN
# ---------------------------------------------------------
# Calculamos cuántos movimientos horizontales y verticales
# separan cada ficha de su posición en el estado meta.
#
# Después sumamos las distancias de todas las fichas.
#
# El espacio vacío (0) NO se incluye.
# ---------------------------------------------------------

def distancia_manhattan(estado, meta):

    # Acumularemos aquí la distancia total.
    distancia_total = 0

    # Recorremos las fichas del 1 al 8.
    for ficha in range(1, 9):

        # Buscamos la posición actual de la ficha.
        posicion_actual = estado.index(ficha)

        # Buscamos dónde debería estar esa misma ficha
        # dentro del estado meta.
        posicion_meta = meta.index(ficha)

        # Convertimos la posición actual en
        # coordenadas de fila y columna.
        fila_actual = posicion_actual // 3
        columna_actual = posicion_actual % 3

        # Hacemos lo mismo con la posición meta.
        fila_meta = posicion_meta // 3
        columna_meta = posicion_meta % 3

        # Calculamos la distancia Manhattan de esta ficha.
        distancia_ficha = (
            abs(fila_actual - fila_meta)
            + abs(columna_actual - columna_meta)
        )

        # Sumamos la distancia de esta ficha al total.
        distancia_total += distancia_ficha

    # Devolvemos la suma de todas las distancias.
    return distancia_total

## Manhattan × 2

In [23]:
# ---------------------------------------------------------
# HEURÍSTICA 3: MANHATTAN × 2
# ---------------------------------------------------------
# Multiplicamos por 2 el valor de la distancia Manhattan.
#
# Esta modificación hará que la heurística sea más
# agresiva y posteriormente analizaremos qué efecto
# tiene sobre A*.
# ---------------------------------------------------------

def manhattan_por_dos(estado, meta):

    # Calculamos Manhattan normalmente y
    # multiplicamos el resultado por 2.
    return 2 * distancia_manhattan(estado, meta)

## Heurística cero

In [24]:
# ---------------------------------------------------------
# HEURÍSTICA 4: HEURÍSTICA CERO
# ---------------------------------------------------------
# Esta heurística devuelve siempre 0 sin importar
# cuál sea la configuración del tablero.
#
# Al utilizarla con A*, tendremos:
#
# f(n) = g(n) + 0
#
# por lo que A* se comportará como búsqueda de
# costo uniforme.
# ---------------------------------------------------------

def heuristica_cero(estado, meta):

    # No importa cuál sea el estado recibido.
    return 0

## Comprobar las cuatro heurísticas

In [25]:
# ---------------------------------------------------------
# COMPROBACIÓN DE LAS HEURÍSTICAS
# ---------------------------------------------------------

# Calculamos cada heurística para el estado inicial.

h_fuera_lugar = fichas_fuera_de_lugar(
    estado_inicial_puzzle,
    estado_meta_puzzle
)

h_manhattan = distancia_manhattan(
    estado_inicial_puzzle,
    estado_meta_puzzle
)

h_manhattan_x2 = manhattan_por_dos(
    estado_inicial_puzzle,
    estado_meta_puzzle
)

h_cero = heuristica_cero(
    estado_inicial_puzzle,
    estado_meta_puzzle
)

# Mostramos los resultados.
print("HEURÍSTICAS DEL ESTADO INICIAL")
print("------------------------------")

print("Fichas fuera de lugar:", h_fuera_lugar)
print("Distancia Manhattan:", h_manhattan)
print("Manhattan × 2:", h_manhattan_x2)
print("Heurística cero:", h_cero)

HEURÍSTICAS DEL ESTADO INICIAL
------------------------------
Fichas fuera de lugar: 4
Distancia Manhattan: 5
Manhattan × 2: 10
Heurística cero: 0


## A* para el 8-puzzle

In [26]:
# ---------------------------------------------------------
# A* PARA EL 8-PUZZLE
# ---------------------------------------------------------
# Esta función resuelve el 8-puzzle utilizando A*.
#
# Recibe:
#   inicio     -> estado inicial del tablero.
#   meta       -> estado que queremos alcanzar.
#   heuristica -> función heurística que A* utilizará.
#
# Devuelve:
#   movimientos       -> secuencia de movimientos realizada.
#   ruta              -> secuencia completa de tableros.
#   costo             -> número de movimientos de la solución.
#   nodos_expandidos  -> cantidad de estados expandidos.
# ---------------------------------------------------------

def a_estrella_puzzle(inicio, meta, heuristica):

    # -----------------------------------------------------
    # FRONTERA
    # -----------------------------------------------------
    # Cada elemento tendrá:
    #
    # (f(n), g(n), contador, estado, movimientos, ruta)
    #
    # donde:
    # g(n) = movimientos realizados desde el inicio.
    # h(n) = estimación proporcionada por la heurística.
    # f(n) = g(n) + h(n).
    #
    # "contador" nos servirá para desempatar elementos
    # que tengan los mismos valores de f(n) y g(n).
    # -----------------------------------------------------

    contador = 0

    # En el estado inicial:
    # g(n) = 0
    # f(n) = h(inicio)
    valor_f_inicial = heuristica(inicio, meta)

    frontera = [
        (
            valor_f_inicial,
            0,
            contador,
            inicio,
            [],
            [inicio]
        )
    ]

    # -----------------------------------------------------
    # MEJORES COSTOS
    # -----------------------------------------------------
    # Guardamos el menor número de movimientos conocido
    # para alcanzar cada configuración del tablero.
    mejores_costos = {
        inicio: 0
    }

    # Contador de estados expandidos.
    nodos_expandidos = 0

    # -----------------------------------------------------
    # BÚSQUEDA
    # -----------------------------------------------------

    while frontera:

        # Extraemos el estado con el menor valor f(n).
        (
            valor_f,
            costo_g,
            _,
            estado_actual,
            movimientos,
            ruta
        ) = heapq.heappop(frontera)

        # Si encontramos el estado meta,
        # devolvemos todos los resultados.
        if estado_actual == meta:
            return (
                movimientos,
                ruta,
                costo_g,
                nodos_expandidos
            )

        # Si ya conocemos una forma más barata de llegar
        # a este estado, ignoramos esta entrada.
        if costo_g > mejores_costos[estado_actual]:
            continue

        # Vamos a generar sus sucesores, por lo que
        # contamos este estado como expandido.
        nodos_expandidos += 1

        # Generamos todos los movimientos legales.
        for movimiento, nuevo_estado in obtener_sucesores(
            estado_actual
        ):

            # Cada movimiento del 8-puzzle cuesta 1.
            nuevo_costo_g = costo_g + 1

            # Solo consideramos este camino si nunca
            # habíamos llegado al nuevo estado o si
            # encontramos una forma más barata.
            if (
                nuevo_estado not in mejores_costos
                or nuevo_costo_g < mejores_costos[nuevo_estado]
            ):

                # Guardamos el nuevo mejor costo.
                mejores_costos[nuevo_estado] = nuevo_costo_g

                # Calculamos la heurística h(n).
                valor_h = heuristica(
                    nuevo_estado,
                    meta
                )

                # Calculamos:
                #
                # f(n) = g(n) + h(n)
                nuevo_valor_f = nuevo_costo_g + valor_h

                # Guardamos la secuencia de movimientos.
                nuevos_movimientos = (
                    movimientos + [movimiento]
                )

                # Guardamos también la secuencia
                # completa de estados.
                nueva_ruta = ruta + [nuevo_estado]

                # Incrementamos el contador de desempate.
                contador += 1

                # Introducimos el nuevo estado
                # en la cola de prioridad.
                heapq.heappush(
                    frontera,
                    (
                        nuevo_valor_f,
                        nuevo_costo_g,
                        contador,
                        nuevo_estado,
                        nuevos_movimientos,
                        nueva_ruta
                    )
                )

    # Si se agota la frontera sin encontrar la meta,
    # el problema no tiene solución desde este estado.
    return None, None, None, nodos_expandidos

## Primera prueba: Manhattan

In [27]:
# ---------------------------------------------------------
# PRUEBA DE A* CON DISTANCIA MANHATTAN
# ---------------------------------------------------------

# Ejecutamos A* utilizando la distancia Manhattan
# como función heurística.
(
    movimientos_manhattan,
    ruta_manhattan,
    costo_manhattan,
    nodos_manhattan
) = a_estrella_puzzle(
    estado_inicial_puzzle,
    estado_meta_puzzle,
    distancia_manhattan
)

# Mostramos los resultados.
print("A* CON DISTANCIA MANHATTAN")
print("--------------------------")

print("Costo de la solución:", costo_manhattan)
print("Nodos expandidos:", nodos_manhattan)

print(
    "Movimientos:",
    " -> ".join(movimientos_manhattan)
)

A* CON DISTANCIA MANHATTAN
--------------------------
Costo de la solución: 5
Nodos expandidos: 5
Movimientos: Arriba -> Arriba -> Izquierda -> Abajo -> Derecha


## Visualizar la solución encontrada

In [28]:
# ---------------------------------------------------------
# VISUALIZACIÓN DE LA SOLUCIÓN ENCONTRADA
# ---------------------------------------------------------
# Mostramos paso a paso los estados recorridos por A*
# utilizando la distancia Manhattan.
# ---------------------------------------------------------

# Recorremos todos los estados de la ruta.
for paso, estado in enumerate(ruta_manhattan):

    # El primer estado corresponde al tablero inicial.
    if paso == 0:
        print("ESTADO INICIAL")

    # Para los demás estados mostramos el movimiento
    # que permitió llegar hasta ellos.
    else:
        print(
            f"\nPASO {paso}: "
            f"{movimientos_manhattan[paso - 1]}"
        )

    # Mostramos el tablero correspondiente.
    mostrar_tablero(estado)

ESTADO INICIAL
2 8 3
1 6 4
7 _ 5

PASO 1: Arriba
2 8 3
1 _ 4
7 6 5

PASO 2: Arriba
2 _ 3
1 8 4
7 6 5

PASO 3: Izquierda
_ 2 3
1 8 4
7 6 5

PASO 4: Abajo
1 2 3
_ 8 4
7 6 5

PASO 5: Derecha
1 2 3
8 _ 4
7 6 5


## Importar time

In [29]:
# Importamos el módulo time para medir
# el tiempo de ejecución de cada búsqueda.
import time

## Función para ejecutar y medir

In [30]:
# ---------------------------------------------------------
# FUNCIÓN PARA EVALUAR UNA HEURÍSTICA
# ---------------------------------------------------------
# Ejecuta A* con una heurística determinada y registra:
#
# - costo de la solución
# - nodos expandidos
# - tiempo de ejecución
# ---------------------------------------------------------

def evaluar_heuristica(nombre, funcion_heuristica):

    # Registramos el instante justo antes
    # de comenzar la búsqueda.
    inicio_tiempo = time.perf_counter()

    # Ejecutamos A* con la heurística recibida.
    movimientos, ruta, costo, nodos = a_estrella_puzzle(
        estado_inicial_puzzle,
        estado_meta_puzzle,
        funcion_heuristica
    )

    # Registramos el instante al terminar.
    fin_tiempo = time.perf_counter()

    # Calculamos el tiempo transcurrido.
    tiempo = fin_tiempo - inicio_tiempo

    # Devolvemos los resultados organizados
    # en un diccionario.
    return {
        "Heurística": nombre,
        "Costo": costo,
        "Nodos expandidos": nodos,
        "Tiempo (s)": tiempo
    }

## Ejecutar los cuatro experimentos

In [31]:
# ---------------------------------------------------------
# COMPARACIÓN DE LAS CUATRO HEURÍSTICAS
# ---------------------------------------------------------

# Ejecutamos A* una vez con cada heurística.
resultados_puzzle = [
    evaluar_heuristica(
        "Fichas fuera de lugar",
        fichas_fuera_de_lugar
    ),

    evaluar_heuristica(
        "Manhattan",
        distancia_manhattan
    ),

    evaluar_heuristica(
        "Manhattan x 2",
        manhattan_por_dos
    ),

    evaluar_heuristica(
        "Heurística cero",
        heuristica_cero
    )
]

# Convertimos los resultados en un DataFrame.
tabla_puzzle = pd.DataFrame(resultados_puzzle)

# Mostramos la tabla.
tabla_puzzle

,Heurística,Costo,Nodos expandidos,Tiempo (s)
0,Fichas fuera de lugar,5,6,0.001988
1,Manhattan,5,5,0.000220
2,Manhattan x 2,5,5,0.000120
3,Heurística cero,5,34,0.001814


## Análisis de resultados - Parte B

Las cuatro configuraciones de A* encontraron una solución con costo 5, correspondiente a cinco movimientos desde el estado inicial hasta el estado meta.

### ¿Qué heurística fue más eficiente?

En términos de nodos expandidos, las heurísticas Manhattan y Manhattan × 2 fueron las más eficientes en este experimento, ya que ambas expandieron únicamente 5 nodos.

La heurística de fichas fuera de lugar expandió 6 nodos, mientras que la heurística cero expandió 34. Esto muestra la utilidad de incorporar información heurística para orientar la búsqueda hacia la meta.

Los tiempos registrados fueron muy pequeños y pueden variar entre ejecuciones debido al entorno de ejecución. Por ello, en este experimento el número de nodos expandidos constituye una medida más estable para comparar la eficiencia de la búsqueda.

### Heurística cero

Cuando h(n) = 0 para todos los estados, A* utiliza:

f(n) = g(n) + 0

Por lo tanto, la búsqueda se comporta como una búsqueda de costo uniforme. En este experimento fue la configuración que más nodos expandió, con 34 estados.

### ¿Manhattan × 2 sigue siendo admisible?

No. Para el estado inicial, la distancia Manhattan tiene un valor de 5, por lo que Manhattan × 2 produce:

h(n) = 2(5) = 10

Sin embargo, el costo óptimo real desde el estado inicial hasta la meta es 5. Por tanto:

h(n) = 10 > h*(n) = 5

La heurística sobreestima el costo real y, por ello, no es admisible.

Aunque A* con Manhattan × 2 encontró en este experimento una solución óptima de costo 5, esto no significa que conserve la garantía de optimalidad. Una heurística no admisible puede encontrar una solución óptima en casos particulares, pero A* deja de tener la garantía general de hacerlo.

## Conclusiones

La práctica permitió comparar el efecto de diferentes estrategias heurísticas sobre el comportamiento de los algoritmos de búsqueda.

En el mapa de carreteras, Voraz, A* y UCS encontraron la misma ruta de 441 km. La búsqueda voraz expandió menos nodos, pero no posee garantía general de optimalidad. A* combinó el costo acumulado g(n) con la estimación h(n), y la heurística utilizada en el mapa fue comprobada como admisible y consistente.

En el 8-puzzle se observó con mayor claridad la influencia de la heurística. Manhattan y Manhattan × 2 expandieron 5 nodos, fichas fuera de lugar expandió 6 y la heurística cero expandió 34.

La heurística Manhattan permitió encontrar la solución óptima orientando eficazmente la búsqueda. En cambio, Manhattan × 2 sobreestimó el costo real desde el estado inicial, por lo que no es admisible y pierde la garantía de optimalidad, aunque en este caso particular también encontró una solución de costo 5.

Los experimentos muestran que una heurística más informativa puede reducir considerablemente el espacio explorado, pero una estimación excesiva puede sacrificar las propiedades teóricas que garantizan una solución óptima.